In [1]:
# 加载根目录
import os

# 直接指定项目根目录的绝对路径
project_root = "d:\\project\\storySummary"
os.chdir(project_root)
print(f"工作目录: {os.getcwd()}")

# 加载配置
from src.config import config

工作目录: d:\project\storySummary


In [3]:
import os
import pathlib
from src.pipeline import NovelToPodcastPipeline

# 确保使用正确的路径分隔符
book_path = pathlib.Path("samples") / "局外人.epub"

# 检查文件是否存在
print(f"文件是否存在: {book_path.exists()}")
print(f"文件路径: {book_path.absolute()}")

# 然后再创建 pipeline 并处理文件
pipeline = NovelToPodcastPipeline(db_path=config.DATABASE_PATH, vector_store_path=config.VECTOR_DB_DIR, 
    api_key=config.DEEPSEEK_API_KEY, user_id="test_user2"
)


await pipeline.process_file(book_path)
print("处理完成")

文件是否存在: True
文件路径: d:\project\storySummary\samples\局外人.epub


AttributeError: property 'chapters' of 'EpubReader' object has no setter

In [ ]:
from src.generation.agents import OutlineAgent
from src.storage.book_repository import  book_repository
from src.core.agents.agent3_interesting_finder import Agent3InterestingFinder

agent3 = Agent3InterestingFinder()
TEST_BOOKID = '66ffedc5-fd54-43ff-acb7-879876243c94'
chunks = book_repository.load_chunks(TEST_BOOKID)
print(f"总共加载了 {len(chunks)} 个文本块")

chunks =await agent3.process_chunks(chunks)
for i, chunk in enumerate(chunks):
    print(chunk.discussion_prompts)

print("处理完成")


In [ ]:
from src.generation.agents import OutlineAgent
from src.storage.book_repository import  book_repository
from src.storage.manuscript_repository import  manuscript_repository
import os

agent = OutlineAgent(api_key=config.DEEPSEEK_API_KEY,debug_mode=True)

TEST_BOOKID = '66ffedc5-fd54-43ff-acb7-879876243c94'
nodes = book_repository.load_nodes(TEST_BOOKID)
chunks = book_repository.load_chunks(TEST_BOOKID)

reference_file = os.path.join("samples", "product","《神的九十亿个名字》.txt")
with open(reference_file, 'r', encoding='utf-8') as f:
    reference_text = f.read()
print(f"参考文本长度: {len(reference_text)}")

def progress_callback(progress):
    print(f"当前进度: {progress}")

story_synopsis, manuscript_outline_json = await agent.build_outline(book_id=TEST_BOOKID, nodes=nodes, chunks=chunks,reference_script=reference_text, progress_callback=progress_callback)
manuscript_repository.save_synopsis(TEST_BOOKID, story_synopsis)
manuscript_repository.save_outline(TEST_BOOKID, manuscript_outline_json)

print("大纲生成完成")
